<a href="https://colab.research.google.com/github/awal015/BoardgamePDFScrapper/blob/main/boardgame_manual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install cleantext
import os
import time
import urllib.parse
from bs4 import BeautifulSoup
import requests
from cleantext import clean

In [2]:
# Google Auth and Sheets libraries (Pre-installed in Colab)
from google.colab import auth
import gspread
from google.auth import default

In [3]:
SPREADSHEET_NAME = "ShakeyT Boardgames"

In [4]:
# Authenticate the user to access Google Sheets
print("🔑 Authenticating your Google account...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

🔑 Authenticating your Google account...


In [5]:
# Open the sheet and grab all names from the first column (Column A)
try:
    sheet = gc.open(SPREADSHEET_NAME).sheet1
    GAME_COLLECTION = sheet.col_values(1)[1:]
    print(f"📊 Successfully loaded {len(GAME_COLLECTION)} games from your sheet!")
except Exception as e:
    print(f"❌ Error loading spreadsheet. Check the name spelling: {e}")
    GAME_COLLECTION = []

📊 Successfully loaded 189 games from your sheet!


In [8]:
# 2. Destination directory in your Google Drive
DOWNLOAD_DIR = "/content/drive/MyDrive/BoardGameManuals"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Standard header to look like a friendly browser request
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def search_archive_pdf(game_name):
    """Searches the 1jour-1jeu rules archive for a game and pulls the English PDF link."""
    # Clean up common naming patterns for better archive hits
    search_query = game_name.strip()
    encoded_query = urllib.parse.quote(search_query)

    # Target the internal search of the rule archive
    search_url = f"https://en.1jour-1jeu.com/rules/search?q={encoded_query}"

    try:
        response = requests.get(search_url, headers=HEADERS)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, "html.parser")

        # Look for links that contain 'cdn.1j1ju.com/medias' (their direct PDF hosting)
        # and prioritize links with 'rulebook' or '-rules' in the filename
        for link in soup.find_all("a", href=True):
            href = link["href"]
            if "cdn.1j1ju.com/medias" in href and ".pdf" in href.lower():
                # Prefer English rules if the filename hints at language
                if "rulebook" in href.lower() or "rules" in href.lower() or "-en" in href.lower():
                    return href

        # Fallback to the very first PDF link found on the search page if specific keywords aren't matched
        for link in soup.find_all("a", href=True):
            href = link["href"]
            if "cdn.1j1ju.com/medias" in href and ".pdf" in href.lower():
                return href

    except Exception as e:
        print(f"⚠️ Archive search error for {game_name}: {e}")
    return None

def download_pdf(url, game_name):
    """Downloads the manual directly into your Google Drive folder."""
    safe_name = clean(game_name).replace(" ", "_")
    file_path = os.path.join(DOWNLOAD_DIR, f"{safe_name}_Rules.pdf")

    try:
        print(f"📥 Downloading manual for {game_name}...")
        response = requests.get(url, headers=HEADERS, stream=True)

        if response.status_code == 200:
            with open(file_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"✅ Saved to Drive: {safe_name}_Rules.pdf")
        else:
            print(f"❌ Download failed for {game_name} (Status: {response.status_code})")
    except Exception as e:
        print(f"⚠️ Error downloading {game_name}: {e}")

# Main Execution Loop
if GAME_COLLECTION:
    for game in GAME_COLLECTION:
        if not game.strip():
            continue

        print(f"\n🔍 Looking for: {game}")
        pdf_url = search_archive_pdf(game)

        if pdf_url:
            download_pdf(pdf_url, game)
        else:
            print(f"❌ Manual not found in archive for: {game}")

        # Polite delay to respect the server hosting the files
        time.sleep(2)

    print("\n🎉 All done! Check the 'BoardGameManuals' folder in your Google Drive.")


🔍 Looking for: 5-Minute Dungeon


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


📥 Downloading manual for 5-Minute Dungeon...
✅ Saved to Drive: minut_dungeon_Rules.pdf

🔍 Looking for: Agemonia
📥 Downloading manual for Agemonia...
✅ Saved to Drive: agemonia_Rules.pdf

🔍 Looking for: Android: Netrunner
📥 Downloading manual for Android: Netrunner...
✅ Saved to Drive: android_netrunn_Rules.pdf

🔍 Looking for: Apiary
📥 Downloading manual for Apiary...
✅ Saved to Drive: apiari_Rules.pdf

🔍 Looking for: Arcadia Quest
📥 Downloading manual for Arcadia Quest...
✅ Saved to Drive: arcadia_quest_Rules.pdf

🔍 Looking for: Arcadia Quest: Beyond the Grave
❌ Manual not found in archive for: Arcadia Quest: Beyond the Grave

🔍 Looking for: Arcadia Quest: Inferno
📥 Downloading manual for Arcadia Quest: Inferno...
✅ Saved to Drive: arcadia_quest_inferno_Rules.pdf

🔍 Looking for: Archduke
📥 Downloading manual for Archduke...
✅ Saved to Drive: archduk_Rules.pdf

🔍 Looking for: Arcs
📥 Downloading manual for Arcs...
✅ Saved to Drive: arc_Rules.pdf

🔍 Looking for: Arcs: Leaders & Lore Pack
